# Week 2 - Day 4: Exploratory Data Analysis

This notebook explores a music dataset with descriptive statistics, data-quality checks, visualizations, outlier review, and correlation analysis.

## 1. Load the dataset

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Resolve the dataset from different possible working directories.
possible_paths = []
for base in [Path.cwd(), *Path.cwd().parents[:3]]:
    possible_paths.extend([
        base / 'data.csv',
        base / 'week2/day4/data.csv',
        base / 'week2/day4/../day4/data.csv',
    ])

for candidate in possible_paths:
    if candidate.exists():
        data_path = candidate
        break
else:
    raise FileNotFoundError('Could not find the music dataset.')

df = pd.read_csv(data_path)

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print('Dataset shape:', df.shape)
df.head()


Dataset shape: (2017, 16)


,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,target,song_title,artist
0,0.0102,0.833,204600,0.434,0.021900,2,0.1650,-8.795,1,0.4310,150.062,4.0,0.286,1,Mask Off,Future
1,0.1990,0.743,326933,0.359,0.006110,1,0.1370,-10.401,1,0.0794,160.083,4.0,0.588,1,Redbone,Childish Gambino
2,0.0344,0.838,185707,0.412,0.000234,2,0.1590,-7.148,1,0.2890,75.044,4.0,0.173,1,Xanny Family,Future
3,0.6040,0.494,199413,0.338,0.510000,5,0.0922,-15.236,1,0.0261,86.468,4.0,0.230,1,Master Of None,Beach House
4,0.1800,0.678,392893,0.561,0.512000,5,0.4390,-11.648,0,0.0694,174.004,4.0,0.904,1,Parallel Lines,Junior Boys


## 2. Inspect the data structure

In [2]:
print(df.shape)
df.info()
print(df.select_dtypes(include=np.number).describe())

(2017, 16)
<class 'pandas.DataFrame'>
RangeIndex: 2017 entries, 0 to 2016
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   acousticness      2017 non-null   float64
 1   danceability      2017 non-null   float64
 2   duration_ms       2017 non-null   int64  
 3   energy            2017 non-null   float64
 4   instrumentalness  2017 non-null   float64
 5   key               2017 non-null   int64  
 6   liveness          2017 non-null   float64
 7   loudness          2017 non-null   float64
 8   mode              2017 non-null   int64  
 9   speechiness       2017 non-null   float64
 10  tempo             2017 non-null   float64
 11  time_signature    2017 non-null   float64
 12  valence           2017 non-null   float64
 13  target            2017 non-null   int64  
 14  song_title        2017 non-null   str    
 15  artist            2017 non-null   str    
dtypes: float64(10), int64(4), str(2)
memory us

## 3. Check missing values and duplicates

In [3]:
missing_values = df.isnull().sum()
print('Missing values:')
print(missing_values)

duplicate_count = df.duplicated().sum()
print('Duplicate rows before cleaning:', duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)

print('Duplicate rows after cleaning:', df.duplicated().sum())
print('Dataset shape after removing duplicates:', df.shape)

Missing values:
acousticness        0
danceability        0
duration_ms         0
energy              0
instrumentalness    0
key                 0
liveness            0
loudness            0
mode                0
speechiness         0
tempo               0
time_signature      0
valence             0
target              0
song_title          0
artist              0
dtype: int64
Duplicate rows before cleaning: 5
Duplicate rows after cleaning: 0
Dataset shape after removing duplicates: (2012, 16)


### Duplicate Handling

The dataset initially contained 5 duplicated rows. These rows were removed because they repeated complete observations and did not add new information. The index was reset after cleaning.

## 4. Target-class distribution

In [4]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='target')
plt.title('Target Class Distribution')
plt.xlabel('Target Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(df['target'].value_counts())

target
1    1015
0     997
Name: count, dtype: int64


/tmp/ipykernel_599553/1164339183.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The target classes are nearly balanced, with 1020 songs in class 1 and 997 songs in class 0. The balance is close enough that no major resampling step is required for a first pass at modeling.

## 5. Histogram analysis

In [5]:
numeric_features = ['danceability', 'energy', 'loudness', 'tempo']
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, feature in zip(axes.flatten(), numeric_features):
    df[feature].hist(ax=ax, bins=20, edgecolor='black')
    ax.set_title(feature.capitalize())
    ax.set_xlabel(feature.capitalize())
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

/tmp/ipykernel_599553/1942906352.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The danceability distribution is spread across much of the range, while energy is concentrated in the middle-to-upper range. Loudness values are mostly negative and fairly broad, and tempo is concentrated around the typical mid-range values with a smaller number of higher values.

## 6. Box plot and IQR-based outlier detection

In [6]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=df['tempo'])
plt.title('Tempo Distribution')
plt.xlabel('Tempo (BPM)')
plt.tight_layout()
plt.show()

q1 = df['tempo'].quantile(0.25)
q3 = df['tempo'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = df[(df['tempo'] < lower_bound) | (df['tempo'] > upper_bound)]

print('Q1:', q1)
print('Q3:', q3)
print('IQR:', iqr)
print('Lower bound:', lower_bound)
print('Upper bound:', upper_bound)
print('Number of tempo outliers:', len(outliers))
tempo_outlier_percentage = len(outliers) / len(df) * 100
print(f'Tempo outlier percentage: {tempo_outlier_percentage:.2f}%')

Q1: 100.16399999999999
Q3: 137.69525
IQR: 37.53125
Lower bound: 43.86712499999999
Upper bound: 193.992125
Number of tempo outliers: 16
Tempo outlier percentage: 0.80%


/tmp/ipykernel_599553/808353326.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The IQR method identified 15 tempo values outside the expected range. This is a small percentage of the dataset, so the values were kept because unusual tempos may reflect real songs rather than data-entry errors.

## 7. Grouped box plot

In [7]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='target', y='energy')
plt.title('Energy Distribution by Target Class')
plt.xlabel('Target Class')
plt.ylabel('Energy')
plt.tight_layout()
plt.show()

/tmp/ipykernel_599553/3449240907.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The median energy is similar for both target classes, and the boxes overlap strongly. That suggests that energy alone does not clearly separate the classes in this dataset.

## 8. Scatter plots for relationships

In [8]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='energy', y='loudness', hue='target', alpha=0.65)
plt.title('Energy vs Loudness by Target Class')
plt.xlabel('Energy')
plt.ylabel('Loudness')
plt.tight_layout()
plt.show()

/tmp/ipykernel_599553/1867363223.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The energy and loudness relationship is positive, and the correlation between these variables is about 0.76. This indicates that songs with higher energy tend to be louder, although there is still substantial overlap between the target classes.

In [9]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='danceability', y='valence', hue='target', alpha=0.65)
plt.title('Danceability vs Valence by Target Class')
plt.xlabel('Danceability')
plt.ylabel('Valence')
plt.tight_layout()
plt.show()

/tmp/ipykernel_599553/2426488427.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The danceability versus valence plot shows a broad overlap between classes. The relationship is weaker than the energy-loudness pattern, so it is useful but not as clearly separating.

## 9. Correlation matrix and heatmap

In [10]:
numeric_df = df.select_dtypes(include=np.number)
correlation_matrix = numeric_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

/tmp/ipykernel_599553/3902316504.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Strongest Correlation Relationships

The strongest positive relationship in the dataset is between energy and loudness, with a correlation of about 0.76. Acousticness also shows a noticeable negative relationship with energy and loudness, which suggests that less acoustic music tends to be more energetic and louder. Danceability and valence are positively related, although the connection is weaker than the energy-loudness pattern. Correlation does not prove causation, but it highlights which features may be useful for later modeling.

In [11]:
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

strongest_pairs = (
    upper_triangle
    .stack()
    .sort_values(key=lambda values: values.abs(), ascending=False)
)

print('Strongest correlation pairs:')
print(strongest_pairs.head(5))

Strongest correlation pairs:
energy            loudness    0.762286
acousticness      energy     -0.647216
                  loudness   -0.561311
danceability      valence     0.442092
instrumentalness  loudness   -0.353337
dtype: float64


## 10. Pairplot for selected features

In [12]:
selected_features = ['energy', 'loudness', 'danceability', 'valence', 'target']

if all(feature in df.columns for feature in selected_features):
    sns.pairplot(df[selected_features], hue='target', corner=True, plot_kws={'alpha': 0.5})
    plt.show()
else:
    print('One or more selected features is missing from the dataset.')

/tmp/ipykernel_599553/2711398686.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Final EDA Conclusions

The dataset contains 2017 rows and 16 columns after removing the index column. There were no missing values, and 5 duplicated rows were removed before the analysis. The target classes were nearly balanced, with 1020 songs in class 1 and 997 songs in class 0. The histograms showed that danceability and tempo were spread across a wide range, while energy and loudness were more concentrated in the middle-to-upper part of their ranges. The IQR method detected 15 tempo outliers, which represented about 0.74% of the dataset. These values were kept because unusual tempos may reflect real songs rather than data-entry problems. The strongest relationship was between energy and loudness, with a correlation of about 0.76, while acousticness also showed meaningful negative relationships with energy and loudness. These connections may be useful for future modeling, although correlation should not be treated as proof of causation.